Fixes applied:
- Background = 0 (standard)
- Foreground classes = 1–5
- Dataset size control
- Train/Val/Test split
- EDA included


# =====================
# CELL 1: INSTALL
# =====================
# !pip install segmentation-models-pytorch

# =====================
# CELL 2: IMPORTS
# =====================

In [1]:
!pip install segmentation-models-pytorch
!pip install albumentations   # for data augmentation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.7 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import cv2
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import segmentation_models_pytorch as smp


# =====================
# CELL 3: CONFIG
# =====================

In [3]:
IMAGE_DIR = "/kaggle/input/datasets/iharshsinha/deepfashion2-top5-processed/processed/train/images"
ANNOT_DIR = "/kaggle/input/datasets/iharshsinha/deepfashion2-top5-processed/processed/train/annos"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256

NUM_CLASSES = 6  # 0 background + 5 classes
MAX_IMAGES = 2000  # 🔥 control dataset size here---not used now

label_map = {
    "short sleeve top": 1,
    "trousers": 2,
    "shorts": 3,
    "long sleeve top": 4,
    "skirt": 5
}

# Save label map
with open("label_map.json", "w") as f:
    json.dump(label_map, f, indent=4)


# =====================
# CELL 4: EDA
# =====================

In [4]:
def compute_class_distribution(annot_dir, max_files=None):
    counter = Counter()
    files = [f for f in os.listdir(annot_dir) if f.endswith(".json")]

    if max_files:
        files = files[:max_files]

    for file in files:
        with open(os.path.join(annot_dir, file)) as f:
            data = json.load(f)

        for k in data:
            if not k.startswith("item"):
                continue

            cat = data[k]["category_name"]
            if cat in label_map:
                counter[cat] += 1

    print("Class Distribution:")
    for k, v in counter.items():
        print(f"{k}: {v}")



# =====================
# CELL 5: MASK GENERATION
# =====================

In [5]:
def polygon_to_mask(polygons, class_id, mask):
    for poly in polygons:
        pts = np.array(poly).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(mask, [pts], class_id)


def generate_mask(json_path, image_shape):
    h, w = image_shape
    mask = np.zeros((h, w), dtype=np.uint8)  # background = 0

    with open(json_path) as f:
        data = json.load(f)

    for k in data:
        if not k.startswith("item"):
            continue

        item = data[k]
        cat = item["category_name"]

        if cat not in label_map:
            continue

        class_id = label_map[cat]
        polygons = item["segmentation"]

        polygon_to_mask(polygons, class_id, mask)

    return mask



# =====================

# CELL 6: DATASET (UPDATED WITH AUGMENTATION)
# =====================

In [6]:
import albumentations as A

class FashionDataset(Dataset):
    def __init__(self, image_dir, annot_dir, files, transform=None):
        self.image_dir = image_dir
        self.annot_dir = annot_dir
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]

        img_path = os.path.join(self.image_dir, img_name)
        json_path = os.path.join(self.annot_dir, img_name.replace(".jpg", ".json"))

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w, _ = image.shape
        mask = generate_mask(json_path, (h, w))

        # 🔥 Augmentation (train only)
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        # Resize
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

        # Normalize
        image = image / 255.0
        image = (image - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225] # new addition, as resnet-34 expects imagenet normalization something
        image = np.transpose(image, (2, 0, 1)).astype(np.float32)

        return torch.tensor(image), torch.tensor(mask).long()


# NEW CELL: DEFINE AUGMENTATIONS

In [7]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
])

# No augmentation for val/test
val_transform = None
test_transform = None




# =====================
# CELL 7: SPLIT DATA (UPDATED for augmentations)
# =====================


* Stage	Setting
* Debug sample_fraction=0.02
* Mid	sample_fraction=0.1
* Final	sample_fraction=0.3–0.4


In [8]:
import random
from torch.utils.data import Subset 

# 🔥 STEP 1: Create ONE shared file list
files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")]

random.seed(42)
random.shuffle(files)

# 🔥 Apply subsampling ONCE
sample_fraction = 0.3   # change this later (0.1, 0.3, etc.)
num_samples = int(len(files) * sample_fraction)
files = files[:num_samples]

print("Total samples used:", len(files))

# 🔥 STEP 2: Split indices
total = len(files)
train_size = int(0.8 * total)
val_size = int(0.1 * total)
test_size = total - train_size - val_size


# 🔥 set seed for torch RNG
seed = 42
g = torch.Generator()
g.manual_seed(seed)

# 🔥 deterministic split
train_indices, val_indices, test_indices = random_split(
    range(total),
    [train_size, val_size, test_size],
    generator=g
)


# 🔥 STEP 3: Create datasets (same files, different transforms)
train_ds_full = FashionDataset(IMAGE_DIR, ANNOT_DIR, files, transform=train_transform)
val_ds_full   = FashionDataset(IMAGE_DIR, ANNOT_DIR, files, transform=None)
# test_ds_full  = FashionDataset(IMAGE_DIR, ANNOT_DIR, files, transform=None)
test_ds_full = val_ds_full

# 🔥 STEP 4: Apply subsets
train_ds = Subset(train_ds_full, train_indices.indices)
val_ds   = Subset(val_ds_full, val_indices.indices)
test_ds  = Subset(test_ds_full, test_indices.indices)

# 🔥 STEP 5: DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=4
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=4
)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")


Total samples used: 43252
Train: 34601, Val: 4325, Test: 4326


In [9]:
# =====================
# SAVE DATA SPLIT (IMPORTANT)
# =====================

train_files = [files[i] for i in train_indices.indices]
val_files   = [files[i] for i in val_indices.indices]
test_files  = [files[i] for i in test_indices.indices]

split = {
    "train": train_files,
    "val": val_files,
    "test": test_files
}

with open("data_split.json", "w") as f:
    json.dump(split, f)

print("Data split saved to data_split.json")

Data split saved to data_split.json
